# Predicting Vaccination Rates Using Behavioral Data

## Roadmap for the Analysis

This notebook follows the standard supervised-learning workflow:
1. prepare features and labels,
2. fit baseline and Bayesian models under cross-validation,
3. evaluate predictions, and
4. summarize Bayesian convergence diagnostics.

Let $y_i \in \{0,1\}$ denote vaccine uptake for respondent $i$, and let $\hat p_i = \Pr(y_i=1 \mid x_i)$ be the model's predicted probability. The main quantities reported later are AUC, Brier score, posterior predictive intervals, and Bayesian convergence statistics such as $\hat R$ and effective sample size (ESS).

In [1]:
import sys
import importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
from config import CONFIG, ensure_output_dirs
from data_utils import load_joined_data, standardize_missing, add_strata_key, make_folds
import model_utils
importlib.reload(model_utils)
run_training_pipeline = model_utils.run_training_pipeline
from eval_utils import summarize_metrics, plot_roc_curves, plot_calibration, write_outputs
print('✓ Modules loaded')

✓ Modules loaded


In [2]:
ensure_output_dirs()
df = load_joined_data()
df = standardize_missing(df)
df = add_strata_key(df, CONFIG['outcomes'])
print(f'Loaded {len(df)} respondents')
print(f'h1n1 rate: {df["h1n1_vaccine"].mean():.3f}, seasonal rate: {df["seasonal_vaccine"].mean():.3f}')

Loaded 26707 respondents
h1n1 rate: 0.212, seasonal rate: 0.466


## Feature Preparation

Before modeling, the notebook standardizes missing values and creates derived predictors. Conceptually, each raw feature $x_{ij}$ is transformed into a model-ready feature $z_{ij}$ by
$$
z_{ij} = 
\begin{cases}
\dfrac{x_{ij} - \mu_j}{\sigma_j}, & \text{if feature } j \text{ is numeric and scaled} \\
\text{one-hot}(x_{ij}), & \text{if feature } j \text{ is categorical} \\
\text{median-imputed}(x_{ij}), & \text{if feature } j \text{ has missing numeric values}
\end{cases}
$$

The goal is to keep the outcome variable fixed while turning survey responses into a matrix $X \in \mathbb{R}^{n \times p}$ that can be used by both classical and Bayesian classifiers.

In [3]:
# Quick test mode: reduce runtime for iterative debugging.
QUICK_TEST_MODE = False

if QUICK_TEST_MODE:
    # Ensure stratification key exists even if cells were run out of order.
    if '_strata_key' not in df.columns:
        df = add_strata_key(df, CONFIG['outcomes'])

    # Downsample while preserving joint-label balance as much as possible.
    target_n = min(len(df), 4000)
    base_n = len(df)
    df = (
        df.groupby('_strata_key', group_keys=False)
          .apply(lambda g: g.sample(max(1, int(round(target_n * len(g) / base_n))), random_state=CONFIG['seed']))
          .reset_index(drop=True)
    )

    # Keep CV/model settings lightweight for quick checks.
    CONFIG['folds'] = 2
    CONFIG['max_bayes_folds'] = 0

    print('Quick test mode enabled')
    print(f"- rows after sampling: {len(df)}")
    print(f"- folds: {CONFIG['folds']}")
    print(f"- bayesian folds: {CONFIG['max_bayes_folds']}")
else:
    print('Quick test mode disabled: using full dataset and full settings')

Quick test mode disabled: using full dataset and full settings


In [4]:
if '_strata_key' not in df.columns:
    df = add_strata_key(df, CONFIG['outcomes'])

folds = make_folds(df, n_folds=CONFIG['folds'], seed=CONFIG['seed'])
print(f'Created {len(folds)} stratified folds')

Created 3 stratified folds


## Cross-Validation Design

Model performance is estimated with stratified $K$-fold cross-validation. The data are partitioned into folds $D_1, \dots, D_K$ so that each fold approximately preserves the joint outcome distribution.

For fold $k$, the model is trained on
$$
D_{-k} = \bigcup_{j \ne k} D_j
$$
and evaluated on $D_k$. The reported performance is then an average over folds and outcomes. This reduces variance in the reported metrics and makes the comparison between logistic regression, random forest, and Bayesian hierarchical models more stable.

In [5]:
training_result = run_training_pipeline(df, folds, CONFIG, verbose=True)
pred_df = training_result['pred_df']
bayes_diagnostics = training_result['bayes_diagnostics']
bayes_effect_summaries = training_result['bayes_effect_summaries']

Bayesian fits run serially per outcome; intra-fit parallelism is controlled by Stan chains/cores.
Fold 1/3
Building...



Building: found in cache, done.Messages from stanc:
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/2100)
Sampling:   0% (2/2100)
Sampling:   5% (101/2100)
Sampling:  10% (200/2100)
Sampling:  14% (300/2100)
Sampling:  19% (400/2100)
Sampling:  24% (500/2100)
Sampling:  29% (600/2100)
Sampling:  31% (651/2100)
Sampling:  33% (702/2100)
Sampling:  38% (801/2100)
Sampling:  43% (900/2100)
Sampling:  48% (1000/2100)
Sampling:  52% (1100/2100)
Sampling:  57% (1200/2100)
Sampling:  62% (1300/2100)
Sampling:  67% (1400/2100)
Sampling:  71% (1500/2100)
Sampling:  76% (1600/2100)
Sampling:  81% (1700/2100)
Sampling:  86% (1800/2100)
Sampling:  90% (1900/2100)
Sampling:  95% (2000/2100)
Sampling: 100% (2100/2100)
Sampling: 100% (2100/2100), done.
Messages received during sampling:
  Gradient evaluation took 0.001244 seconds
  1000 transitions using 10 leapfrog steps per transition would take 12.44 sec

Building...



Building: found in cache, done.Messages from stanc:
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/2100)
Sampling:   0% (2/2100)
Sampling:   5% (101/2100)
Sampling:  10% (200/2100)
Sampling:  14% (300/2100)
Sampling:  19% (400/2100)
Sampling:  24% (500/2100)
Sampling:  29% (600/2100)
Sampling:  31% (651/2100)
Sampling:  33% (702/2100)
Sampling:  38% (801/2100)
Sampling:  43% (900/2100)
Sampling:  48% (1000/2100)
Sampling:  52% (1100/2100)
Sampling:  57% (1200/2100)
Sampling:  62% (1300/2100)
Sampling:  67% (1400/2100)
Sampling:  71% (1500/2100)
Sampling:  76% (1600/2100)
Sampling:  81% (1700/2100)
Sampling:  86% (1800/2100)
Sampling:  90% (1900/2100)
Sampling:  95% (2000/2100)
Sampling: 100% (2100/2100)
Sampling: 100% (2100/2100), done.
Messages received during sampling:
  Gradient evaluation took 0.001129 seconds
  1000 transitions using 10 leapfrog steps per transition would take 11.29 sec

Fold 2/3
Fold 3/3
✓ Training complete, collected 124634 predictions
✓ Bayesian fits completed: 2
✓ Bayesian prior scale in use (for sensitivity analysis): 1.0


## Model Definitions

The notebook compares three predictors for $\Pr(y_i=1 \mid x_i)$:

- Logistic ridge regression:
$$
\Pr(y_i=1 \mid x_i) = \sigma(\beta_0 + x_i^\top \beta), \quad \sigma(t) = \frac{1}{1 + e^{-t}}
$$

- Random forest: a nonparametric ensemble that averages many decision trees, so the final probability estimate is the mean of tree-level class probabilities.

- Bayesian hierarchical logistic regression:
$$
\Pr(y_i=1 \mid x_i, g_i) = \sigma(\alpha + x_i^\top \beta + u_{g_i}), \qquad u_g \sim \mathcal{N}(0, \sigma_u^2)
$$

The random intercept $u_g$ lets the baseline risk vary by geographic region while borrowing strength across groups through the prior on $\sigma_u$.

In [6]:
metrics = summarize_metrics(pred_df)
print(metrics.to_string())
write_outputs(pred_df, metrics)
print('✓ Metrics saved')

if bayes_diagnostics:
    bayes_diag_df = pd.DataFrame(bayes_diagnostics)
    print('\nBayesian convergence diagnostics (target: R_hat near 1.00, strong ESS):')
    print(bayes_diag_df.to_string(index=False))

if bayes_effect_summaries:
    bayes_effect_df = pd.DataFrame(bayes_effect_summaries)
    print('\nGroup-level random effect uncertainty (sigma_u):')
    print(bayes_effect_df.to_string(index=False))

if {'pred_ci_lower', 'pred_ci_upper'}.issubset(pred_df.columns):
    bayes_pred = pred_df[pred_df['model'] == 'bayesian_hierarchical'].copy()
    if not bayes_pred.empty:
        mean_ci_width = (bayes_pred['pred_ci_upper'] - bayes_pred['pred_ci_lower']).mean()
        print(f"\nBayesian posterior predictive check: mean 95% CI width = {mean_ci_width:.4f}")

                   model           outcome       auc     brier        n  macro_auc  macro_brier
0  bayesian_hierarchical      h1n1_vaccine  0.833236  0.118604   8903.0   0.839316     0.138813
1  bayesian_hierarchical  seasonal_vaccine  0.845397  0.159022   8903.0   0.839316     0.138813
2         logistic_ridge      h1n1_vaccine  0.871037  0.107443  26707.0   0.873421     0.124415
3         logistic_ridge  seasonal_vaccine  0.875805  0.141388  26707.0   0.873421     0.124415
4          random_forest      h1n1_vaccine  0.870250  0.111659  26707.0   0.872590     0.129708
5          random_forest  seasonal_vaccine  0.874929  0.147758  26707.0   0.872590     0.129708
✓ Metrics saved

Bayesian convergence diagnostics (target: R_hat near 1.00, strong ESS):
 fold          outcome  max_rhat  min_bulk_ess  min_tail_ess  divergences  treedepth_saturated
    1     h1n1_vaccine  1.001272    507.900635    507.900635            0                    0
    1 seasonal_vaccine  1.002475    350.939535   

## Evaluation Metrics

The notebook reports two main scoring rules:

1. Area under the ROC curve (AUC), which summarizes ranking quality by integrating the true-positive rate over the false-positive rate.
2. Brier score, which measures the mean squared error of probabilistic predictions:
$$
\text{Brier}(\hat p) = \frac{1}{n} \sum_{i=1}^{n} (\hat p_i - y_i)^2
$$

Lower Brier scores indicate better calibrated probabilities. AUC focuses on discrimination, while Brier score focuses on calibration and overall probability accuracy.

In [7]:
plot_roc_curves(pred_df, CONFIG['paths']['roc_plot'])
plot_calibration(pred_df, CONFIG['paths']['calibration_plot'])
print('✓ Plots generated')

✓ Plots generated


## Bayesian Diagnostics Snapshot

This section aligns with the analysis plan by checking convergence and uncertainty outputs from MCMC.

- Target convergence: max R-hat close to 1.00 and sufficiently large ESS.
- Posterior predictive uncertainty: 95% credible interval width for predicted probabilities.
- Group-level uncertainty: posterior interval for the random-effect scale parameter (sigma_u).

In [8]:
from pathlib import Path

R_HAT_THRESHOLD = 1.01
ESS_THRESHOLD = 400

if not bayes_diagnostics:
    print('No Bayesian diagnostics available. Increase CONFIG["max_bayes_folds"] and rerun training.')
else:
    bayes_diag_df = pd.DataFrame(bayes_diagnostics).copy()

    def resolve_metric_col(df, aliases):
        for name in aliases:
            if name in df.columns:
                return name
        return None

    metric_aliases = {
        'rhat': ['rhat_max', 'max_rhat'],
        'ess_bulk': ['ess_bulk_min', 'min_bulk_ess'],
        'ess_tail': ['ess_tail_min', 'min_tail_ess'],
        'divergences': ['divergences', 'n_divergent'],
        'treedepth': ['treedepth_saturated', 'n_max_treedepth'],
    }

    resolved = {k: resolve_metric_col(bayes_diag_df, v) for k, v in metric_aliases.items()}

    for key, col in resolved.items():
        if col is not None:
            bayes_diag_df[col] = pd.to_numeric(bayes_diag_df[col], errors='coerce')

    rhat_series = bayes_diag_df[resolved['rhat']] if resolved['rhat'] else pd.Series(np.nan, index=bayes_diag_df.index)
    ess_bulk_series = bayes_diag_df[resolved['ess_bulk']] if resolved['ess_bulk'] else pd.Series(np.nan, index=bayes_diag_df.index)
    ess_tail_series = bayes_diag_df[resolved['ess_tail']] if resolved['ess_tail'] else pd.Series(np.nan, index=bayes_diag_df.index)
    divergences_series = bayes_diag_df[resolved['divergences']] if resolved['divergences'] else pd.Series(0, index=bayes_diag_df.index)
    treedepth_series = bayes_diag_df[resolved['treedepth']] if resolved['treedepth'] else pd.Series(0, index=bayes_diag_df.index)

    bayes_diag_df['rhat_ok'] = rhat_series <= R_HAT_THRESHOLD
    bayes_diag_df['ess_bulk_ok'] = ess_bulk_series >= ESS_THRESHOLD
    bayes_diag_df['ess_tail_ok'] = ess_tail_series >= ESS_THRESHOLD
    bayes_diag_df['hmc_ok'] = (divergences_series.fillna(0) == 0) & (treedepth_series.fillna(0) == 0)
    bayes_diag_df['all_checks_pass'] = (
        bayes_diag_df['rhat_ok']
        & bayes_diag_df['ess_bulk_ok']
        & bayes_diag_df['ess_tail_ok']
        & bayes_diag_df['hmc_ok']
    )

    bayes_diag_df['rhat_metric'] = rhat_series
    bayes_diag_df['ess_bulk_metric'] = ess_bulk_series
    bayes_diag_df['ess_tail_metric'] = ess_tail_series
    bayes_diag_df['divergences_metric'] = divergences_series
    bayes_diag_df['treedepth_metric'] = treedepth_series

    display_cols = [
        'fold', 'outcome', 'rhat_metric', 'ess_bulk_metric', 'ess_tail_metric',
        'divergences_metric', 'treedepth_metric', 'all_checks_pass'
    ]
    display_cols = [c for c in display_cols if c in bayes_diag_df.columns]

    print('Bayesian diagnostics per fit:')
    display(bayes_diag_df[display_cols].sort_values(['fold', 'outcome']).reset_index(drop=True))

    snapshot = {
        'n_fits': int(len(bayes_diag_df)),
        'all_checks_pass_rate': float(bayes_diag_df['all_checks_pass'].mean()),
        'max_rhat_observed': float(bayes_diag_df['rhat_metric'].max()),
        'min_ess_bulk_observed': float(bayes_diag_df['ess_bulk_metric'].min()),
        'min_ess_tail_observed': float(bayes_diag_df['ess_tail_metric'].min()),
        'total_divergences': int(bayes_diag_df['divergences_metric'].fillna(0).sum()),
        'total_treedepth_saturated': int(bayes_diag_df['treedepth_metric'].fillna(0).sum()),
    }

    if {'pred_ci_lower', 'pred_ci_upper'}.issubset(pred_df.columns):
        bayes_pred = pred_df[pred_df['model'] == 'bayesian_hierarchical'].copy()
        if not bayes_pred.empty:
            ci_width = bayes_pred['pred_ci_upper'] - bayes_pred['pred_ci_lower']
            snapshot['mean_pred_ci_width'] = float(ci_width.mean())
            snapshot['median_pred_ci_width'] = float(ci_width.median())

    if bayes_effect_summaries:
        sigma_df = pd.DataFrame(bayes_effect_summaries)
        sigma_df = sigma_df.dropna(subset=['group_sd_mean']) if 'group_sd_mean' in sigma_df.columns else sigma_df
        if not sigma_df.empty and 'group_sd_mean' in sigma_df.columns:
            snapshot['mean_group_sd'] = float(pd.to_numeric(sigma_df['group_sd_mean'], errors='coerce').mean())

    snapshot_df = pd.DataFrame([snapshot])
    print('\nBayesian diagnostics snapshot:')
    display(snapshot_df)

    out_dir = Path(CONFIG['paths']['metrics']).parent
    out_dir.mkdir(parents=True, exist_ok=True)
    snapshot_path = out_dir / 'bayesian_diagnostics_snapshot.csv'
    details_path = out_dir / 'bayesian_diagnostics_per_fit.csv'
    snapshot_df.to_csv(snapshot_path, index=False)
    bayes_diag_df.to_csv(details_path, index=False)
    print(f'Saved snapshot: {snapshot_path}')
    print(f'Saved per-fit diagnostics: {details_path}')

Bayesian diagnostics per fit:


,fold,outcome,rhat_metric,ess_bulk_metric,ess_tail_metric,divergences_metric,treedepth_metric,all_checks_pass
0,1,h1n1_vaccine,1.001272,507.900635,507.900635,0,0,True
1,1,seasonal_vaccine,1.002475,350.939535,350.939535,0,1,False



Bayesian diagnostics snapshot:


,n_fits,all_checks_pass_rate,max_rhat_observed,min_ess_bulk_observed,min_ess_tail_observed,total_divergences,total_treedepth_saturated,mean_pred_ci_width,median_pred_ci_width,mean_group_sd
0,2,0.5,1.002475,350.939535,350.939535,0,1,0.05454,0.052389,0.056785


Saved snapshot: /home/silas-falde/Documents/GitHub/datasci-451-final-project/outputs/metrics/bayesian_diagnostics_snapshot.csv
Saved per-fit diagnostics: /home/silas-falde/Documents/GitHub/datasci-451-final-project/outputs/metrics/bayesian_diagnostics_per_fit.csv


## Bayesian Diagnostics in Plain Language

The Bayesian fit is summarized with three common convergence checks:

- $\hat R$ compares between-chain and within-chain variation. Values near 1.0 suggest the chains are mixing well.
- ESS (effective sample size) estimates how many independent draws the posterior sample is worth after accounting for autocorrelation.
- Divergences and treedepth saturation flag numerical problems in Hamiltonian Monte Carlo.

In a simple two-chain setting, if chain means are $\bar\theta_1, \bar\theta_2$ and within-chain variance is $W$, then a schematic version of the potential scale reduction factor is
$$
\hat R = \sqrt{\frac{\widehat{\mathrm{Var}}^+(\theta)}{W}}
$$

The notebook also reports posterior uncertainty for the random-effect scale parameter $\sigma_u$ and the average width of predictive credible intervals. These are useful for explaining both model fit and uncertainty in the final report.